# 01 — Data Understanding

**Objective:** profile every Olist source table before analysis: row counts,
grain, keys, missing values, value ranges and known quality issues.

**Business context:** Olist is a Brazilian marketplace connecting small
sellers to large retailers' sites. Each table represents one part of the
order lifecycle. Understanding the grain of each table is essential —
a wrong join (e.g. order x items x payments in one flat table) silently
inflates every metric.

In [1]:
import sys
from pathlib import Path

# Make the repo's src/ package importable when running from notebooks/.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data_loading as dl
from src.config import get_engine
from src.visualization import PALETTE, set_style

set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
engine = get_engine()


## Row counts and grain

In [2]:
tables = {
    "customers":            "one row per customer_id (order-level record)",
    "orders":               "one row per order",
    "order_items":          "one row per (order_id, order_item_id)",
    "order_payments":       "one row per (order_id, payment_sequential)",
    "order_reviews":        "one row per review record (review_id NOT unique)",
    "products":             "one row per product_id",
    "sellers":              "one row per seller_id",
    "geolocation":          "one row per (zip prefix, lat, lng) point",
    "category_translation": "one row per PT category name",
}
rows = [
    {"table": t, "grain": g,
     "rows": dl.query(f"SELECT COUNT(*) n FROM raw.{t}", engine)["n"].iloc[0]}
    for t, g in tables.items()
]
pd.DataFrame(rows)

,table,grain,rows
0,customers,one row per customer_id (order-level record),99441
1,orders,one row per order,99441
2,order_items,"one row per (order_id, order_item_id)",112650
3,order_payments,"one row per (order_id, payment_sequential)",103886
4,order_reviews,one row per review record (review_id NOT unique),99224
5,products,one row per product_id,32951
6,sellers,one row per seller_id,3095
7,geolocation,"one row per (zip prefix, lat, lng) point",1000163
8,category_translation,one row per PT category name,71


## Orders table - status mix and date coverage

In [3]:
dl.query("""
    SELECT order_status, COUNT(*) AS orders,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pct
    FROM raw.orders GROUP BY order_status ORDER BY orders DESC
""", engine)

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [4]:
dl.query("""
    SELECT MIN(order_purchase_timestamp) AS first_purchase,
           MAX(order_purchase_timestamp) AS last_purchase,
           MIN(order_delivered_customer_date) AS first_delivery,
           MAX(order_estimated_delivery_date) AS last_estimate
    FROM raw.orders
""", engine)

,first_purchase,last_purchase,first_delivery,last_estimate
0,2016-09-04 21:15:19,2018-10-17 17:30:18,2016-10-11 13:46:32,2018-11-12


**Observation:** purchases span **Sep-2016 to Oct-2018**. Any recency/
retention statement must be made relative to this window — the data is
historical, so "now" is meaningless.

## Missing values

In [5]:
missing = dl.query("""
    SELECT 'orders.delivered_customer_date' AS col,
           COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS nulls
    FROM raw.orders
    UNION ALL
    SELECT 'orders.delivered_carrier_date',
           COUNT(*) FILTER (WHERE order_delivered_carrier_date IS NULL)
    FROM raw.orders
    UNION ALL
    SELECT 'orders.approved_at',
           COUNT(*) FILTER (WHERE order_approved_at IS NULL) FROM raw.orders
    UNION ALL
    SELECT 'products.category_name',
           COUNT(*) FILTER (WHERE product_category_name IS NULL) FROM raw.products
    UNION ALL
    SELECT 'reviews.comment_message',
           COUNT(*) FILTER (WHERE review_comment_message IS NULL)
    FROM raw.order_reviews
    UNION ALL
    SELECT 'reviews.comment_title',
           COUNT(*) FILTER (WHERE review_comment_title IS NULL)
    FROM raw.order_reviews
""", engine)
missing

,col,nulls
0,products.category_name,610
1,orders.delivered_customer_date,2965
2,orders.approved_at,160
3,orders.delivered_carrier_date,1783
4,reviews.comment_title,87656
5,reviews.comment_message,58247


Missing **delivery timestamps** are expected for non-delivered orders
(shipped/canceled/etc.) — records are excluded per-metric, not deleted.
Missing review comments are normal (most buyers leave only a score).

## Customer identity - the most important modelling fact

In [6]:
dl.query("""
    SELECT COUNT(DISTINCT customer_id)         AS customer_ids,
           COUNT(DISTINCT customer_unique_id)  AS real_unique_customers,
           (SELECT COUNT(*) FROM (
               SELECT customer_unique_id FROM raw.customers
               GROUP BY customer_unique_id HAVING COUNT(*) > 1) r)
                                              AS customers_with_multiple_ids
    FROM raw.customers
""", engine)

,customer_ids,real_unique_customers,customers_with_multiple_ids
0,99441,96096,2997


`customer_id` is generated **per order**; `customer_unique_id` is the
same person across orders. Every customer-level metric in this project uses
`customer_unique_id`.

## Known data-quality issues (from sql/01_data_quality_checks.sql)

* 789 duplicated `review_id`s and 547 orders with multiple review rows
  -> staging keeps ONE review per order (latest creation date).
* 8 delivered orders have no delivery timestamp; 6 non-delivered orders do.
* 23 orders delivered "before" carrier handover (timestamp noise).
* 9 zero-value payments; 1 order with no payment rows.
* 246 delivered orders where payment total differs from items+freight
  by >R$1 -> we therefore define revenue as items+freight, not payments.
* 610 products lack a category name; 13 categories lack English
  translation -> mapped to 'unknown'.

## Conclusion

The dataset is internally consistent (zero orphaned keys, zero invalid
scores/dates at the grain level) but has quirks that must be handled
per-metric. The staging layer in `sql/02_staging_tables.sql` encodes
all of these decisions explicitly.